## 1. Setup and Data Loading

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr, kruskal
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✓ Libraries loaded")

In [ ]:
def load_nde_data(structured_dir: Path) -> pd.DataFrame:
    """Load NDE records and extract fields relevant to Threefold Path analysis.
    
    Extracts stage indicators for:
    - Stage 1 (Passage): OBE, tunnel, light, movement
    - Stage 2 (Arrival): Environment, beings, greeting
    - Stage 3 (Life Review): Review occurrence, judgment
    - Stage 4 (Return): Return type, agency, willingness, boundary
    """
    records = []
    
    for file in structured_dir.glob("*.json"):
        with open(file, 'r', encoding='utf-8') as f:
            record = json.load(f)
        
        data = {'nde_code': record.get('title', 'unknown'),
                'dataset': record.get('dataset', 'unknown')}
        
        analysis = record.get('analysis', {}) or {}
        
        # === DEMOGRAPHICS ===
        demo = analysis.get('person_demographics', {}) or {}
        data['age_at_nde'] = demo.get('age_at_nde')
        data['gender'] = demo.get('gender', 'not_mentioned')
        data['religious_belief_at_nde'] = demo.get('religious_belief_at_nde', 'not_mentioned')
        
        # === NDE CIRCUMSTANCES ===
        circumstances = analysis.get('nde_circumstances', {}) or {}
        data['cause'] = circumstances.get('cause', 'not_mentioned')
        data['setting'] = circumstances.get('setting', 'not_mentioned')
        
        # === STAGE 1: PASSAGE ===
        passage = analysis.get('passage', {}) or {}
        separation = passage.get('separation', {}) or {}
        transition = passage.get('transition', {}) or {}
        arrival = passage.get('arrival', {}) or {}
        
        # OBE
        data['obe_occurrence'] = separation.get('obe_occurrence', 'not_mentioned')
        data['has_obe'] = data['obe_occurrence'] in ['yes_verified', 'yes_unverified']
        
        # Tunnel
        data['tunnel_experience'] = transition.get('tunnel_experience', 'not_mentioned')
        data['has_tunnel'] = data['tunnel_experience'] in ['classic_tunnel', 'tunnel_like', 'void_darkness']
        
        # Light encounter
        data['light_encounter'] = arrival.get('light_encounter', 'not_mentioned')
        data['has_light'] = data['light_encounter'] not in ['not_mentioned', 'no_light']
        
        # Movement sensation
        data['movement_sensation'] = transition.get('movement_sensation', 'not_mentioned')
        data['has_movement'] = data['movement_sensation'] not in ['not_mentioned', 'no_movement']
        
        # Stage 1 composite
        data['stage1_passage'] = data['has_obe'] or data['has_tunnel'] or data['has_movement']
        data['stage1_elements'] = sum([data['has_obe'], data['has_tunnel'], data['has_light'], data['has_movement']])
        
        # === STAGE 2: ARRIVAL ===
        # Environment
        data['environment_type'] = arrival.get('environment_type', 'not_mentioned')
        data['has_environment'] = data['environment_type'] not in ['not_mentioned', 'undefined_space']
        
        # Beings
        being_ids = arrival.get('being_identifications', [])
        if not isinstance(being_ids, list):
            being_ids = [being_ids] if being_ids else []
        data['being_identifications'] = being_ids
        data['has_beings'] = len(being_ids) > 0
        
        # Greeting
        greeting_types = arrival.get('greeting_types', [])
        if not isinstance(greeting_types, list):
            greeting_types = [greeting_types] if greeting_types else []
        data['greeting_types'] = greeting_types
        data['has_greeting'] = len(greeting_types) > 0
        
        # Sense of belonging
        data['sense_of_belonging'] = arrival.get('sense_of_belonging', 'not_mentioned')
        data['has_belonging'] = data['sense_of_belonging'] in ['strong', 'moderate', 'yes']
        
        # Stage 2 composite
        data['stage2_arrival'] = data['has_environment'] or data['has_beings'] or data['has_light']
        data['stage2_elements'] = sum([data['has_environment'], data['has_beings'], data['has_greeting'], data['has_belonging']])
        
        # === STAGE 3: LIFE REVIEW ===
        review = analysis.get('life_review', {}) or {}
        
        data['life_review_occurrence'] = review.get('occurrence', 'not_mentioned')
        data['has_life_review'] = data['life_review_occurrence'] in ['extensive', 'brief']
        
        data['judgment_source'] = review.get('judgment_source', 'not_mentioned')
        data['judgment_intensity'] = review.get('judgment_intensity', 'not_specified')
        data['perspective_of_others'] = review.get('perspective_of_others', 'not_mentioned')
        data['has_perspective_shift'] = data['perspective_of_others'] in ['yes_full', 'yes_partial']
        
        # Stage 3 composite
        data['stage3_review'] = data['has_life_review']
        data['stage3_elements'] = sum([data['has_life_review'], data['has_perspective_shift']])
        
        # === STAGE 4: BOUNDARY/RETURN ===
        return_data = analysis.get('return_integration', {}) or {}
        
        data['return_experience'] = return_data.get('return_experience', 'not_mentioned')
        data['has_return'] = data['return_experience'] not in ['not_mentioned', 'no_return_memory']
        
        # NEW SPLIT FIELDS
        data['return_agency'] = return_data.get('return_agency', 'not_mentioned')
        data['return_willingness'] = return_data.get('return_willingness', 'not_mentioned')
        data['boundary_type'] = return_data.get('boundary_type', 'not_mentioned')
        data['has_boundary'] = data['boundary_type'] not in ['not_mentioned', 'no_boundary']
        
        # Stage 4 composite
        data['stage4_return'] = data['has_return'] or data['has_boundary']
        data['stage4_elements'] = sum([data['has_return'], data['has_boundary']])
        
        # === OVERALL METRICS ===
        data['stages_present'] = sum([data['stage1_passage'], data['stage2_arrival'], 
                                      data['stage3_review'], data['stage4_return']])
        data['total_elements'] = data['stage1_elements'] + data['stage2_elements'] + \
                                 data['stage3_elements'] + data['stage4_elements']
        
        # === EXPERIENCE QUALITY ===
        quality = analysis.get('experience_quality', {}) or {}
        data['overall_valence'] = quality.get('overall_valence', 'not_mentioned')
        data['reality_assessment'] = quality.get('reality_assessment', 'not_mentioned')
        
        records.append(data)
    
    return pd.DataFrame(records)

# Load data
print("Loading NDE records...")
structured_dir = Path("../structured")
df = load_nde_data(structured_dir)

print(f"\n✓ Loaded {len(df):,} total records")
print(f"  NDERF: {(df['dataset'] == 'nderf').sum():,}")
print(f"  IANDS: {(df['dataset'] == 'iands').sum():,}")
print(f"  Features: {df.shape[1]}")

## 2. Data Quality Overview

In [ ]:
print("="*70)
print("DATA QUALITY: Stage Element Distributions")
print("="*70)

# Stage 1 elements
print("\n📊 STAGE 1 - PASSAGE Elements:")
print(f"  Has OBE: {df['has_obe'].sum():,} ({df['has_obe'].mean()*100:.1f}%)")
print(f"  Has tunnel: {df['has_tunnel'].sum():,} ({df['has_tunnel'].mean()*100:.1f}%)")
print(f"  Has light: {df['has_light'].sum():,} ({df['has_light'].mean()*100:.1f}%)")
print(f"  Has movement: {df['has_movement'].sum():,} ({df['has_movement'].mean()*100:.1f}%)")

# Stage 2 elements
print("\n📊 STAGE 2 - ARRIVAL Elements:")
print(f"  Has environment: {df['has_environment'].sum():,} ({df['has_environment'].mean()*100:.1f}%)")
print(f"  Has beings: {df['has_beings'].sum():,} ({df['has_beings'].mean()*100:.1f}%)")
print(f"  Has greeting: {df['has_greeting'].sum():,} ({df['has_greeting'].mean()*100:.1f}%)")
print(f"  Has belonging: {df['has_belonging'].sum():,} ({df['has_belonging'].mean()*100:.1f}%)")

# Stage 3 elements
print("\n📊 STAGE 3 - LIFE REVIEW Elements:")
print(f"  Has life review: {df['has_life_review'].sum():,} ({df['has_life_review'].mean()*100:.1f}%)")
print(f"  Has perspective shift: {df['has_perspective_shift'].sum():,} ({df['has_perspective_shift'].mean()*100:.1f}%)")

# Stage 4 elements
print("\n📊 STAGE 4 - RETURN Elements:")
print(f"  Has return memory: {df['has_return'].sum():,} ({df['has_return'].mean()*100:.1f}%)")
print(f"  Has boundary: {df['has_boundary'].sum():,} ({df['has_boundary'].mean()*100:.1f}%)")

---

## 3. Test 1: Stage Occurrence Rates

**Question**: What proportion of NDEs include each stage of the journey?

In [ ]:
print("="*70)
print("TEST 1: STAGE OCCURRENCE RATES")
print("="*70)

# Stage occurrence
stages = {
    'Stage 1: Passage': df['stage1_passage'].mean() * 100,
    'Stage 2: Arrival': df['stage2_arrival'].mean() * 100,
    'Stage 3: Life Review': df['stage3_review'].mean() * 100,
    'Stage 4: Return': df['stage4_return'].mean() * 100
}

print("\n📊 Stage Occurrence Rates:")
for stage, rate in stages.items():
    count = int(len(df) * rate / 100)
    print(f"  {stage}: {count:,} ({rate:.1f}%)")

# Number of stages per experience
print("\n📊 Number of Stages Per Experience:")
stage_counts = df['stages_present'].value_counts().sort_index()
for n_stages, count in stage_counts.items():
    print(f"  {n_stages} stages: {count:,} ({count/len(df)*100:.1f}%)")

# Mean stages
print(f"\n  Mean stages present: {df['stages_present'].mean():.2f}")
print(f"  Median stages present: {df['stages_present'].median():.1f}")

In [ ]:
# Visualization: Stage Occurrence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of stage rates
ax1 = axes[0]
stage_names = list(stages.keys())
stage_rates = list(stages.values())
colors = ['#1a9850', '#91cf60', '#d9ef8b', '#fee08b']
bars = ax1.bar(range(4), stage_rates, color=colors)
ax1.set_xticks(range(4))
ax1.set_xticklabels(['Passage', 'Arrival', 'Life Review', 'Return'])
ax1.set_ylabel('Percentage of NDEs')
ax1.set_title('Stage Occurrence Rates')
ax1.set_ylim(0, 100)
for i, v in enumerate(stage_rates):
    ax1.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

# Distribution of stages per experience
ax2 = axes[1]
stage_dist = df['stages_present'].value_counts().sort_index()
ax2.bar(stage_dist.index, stage_dist.values, color='steelblue')
ax2.set_xlabel('Number of Stages Present')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Stage Completeness')
ax2.set_xticks([0, 1, 2, 3, 4])

plt.tight_layout()
plt.show()

---

## 4. Test 2: Sequential Order Validation

**Hypothesis**: Stages follow a predictable sequence (Passage → Arrival → Review → Return).

**Test**: For experiences with multiple stages, check if later stages are contingent on earlier stages.

In [ ]:
print("="*70)
print("TEST 2: SEQUENTIAL ORDER VALIDATION")
print("="*70)

# Create stage pattern string
df['stage_pattern'] = (df['stage1_passage'].astype(int).astype(str) + 
                       df['stage2_arrival'].astype(int).astype(str) + 
                       df['stage3_review'].astype(int).astype(str) + 
                       df['stage4_return'].astype(int).astype(str))

# Expected sequential patterns (building up)
sequential_patterns = {
    '0000': 'No stages',
    '1000': 'Passage only',
    '1100': 'Passage → Arrival',
    '1110': 'Passage → Arrival → Review',
    '1111': 'Complete journey'
}

print("\n📊 Stage Pattern Distribution:")
pattern_counts = df['stage_pattern'].value_counts().head(15)
for pattern, count in pattern_counts.items():
    label = sequential_patterns.get(pattern, f'Pattern {pattern}')
    pct = count / len(df) * 100
    # Mark if sequential
    is_seq = pattern in sequential_patterns
    marker = '✓' if is_seq else '✗'
    print(f"  {marker} {pattern} ({label}): {count:,} ({pct:.1f}%)")

# Calculate sequential vs non-sequential
sequential = df['stage_pattern'].isin(sequential_patterns.keys()).sum()
non_sequential = len(df) - sequential

print(f"\n📈 SEQUENTIAL PATTERNS:")
print(f"  Sequential (building up): {sequential:,} ({sequential/len(df)*100:.1f}%)")
print(f"  Non-sequential (skipping): {non_sequential:,} ({non_sequential/len(df)*100:.1f}%)")

In [ ]:
# Conditional probability analysis
print("\n📊 CONDITIONAL PROBABILITIES (Later Stage | Earlier Stage):")
print("-" * 60)

# P(Arrival | Passage)
passage_cases = df[df['stage1_passage']]
p_arrival_given_passage = passage_cases['stage2_arrival'].mean() * 100
print(f"  P(Arrival | Passage) = {p_arrival_given_passage:.1f}%")

# P(Review | Arrival)
arrival_cases = df[df['stage2_arrival']]
p_review_given_arrival = arrival_cases['stage3_review'].mean() * 100
print(f"  P(Review | Arrival) = {p_review_given_arrival:.1f}%")

# P(Return | Review)
review_cases = df[df['stage3_review']]
p_return_given_review = review_cases['stage4_return'].mean() * 100
print(f"  P(Return | Review) = {p_return_given_review:.1f}%")

# Compare to unconditional
print("\n📊 UNCONDITIONAL PROBABILITIES (baseline):")
print(f"  P(Arrival) = {df['stage2_arrival'].mean()*100:.1f}%")
print(f"  P(Review) = {df['stage3_review'].mean()*100:.1f}%")
print(f"  P(Return) = {df['stage4_return'].mean()*100:.1f}%")

# Lift analysis
print("\n📈 LIFT (Conditional / Unconditional):")
lift_arrival = p_arrival_given_passage / (df['stage2_arrival'].mean() * 100)
lift_review = p_review_given_arrival / (df['stage3_review'].mean() * 100)
lift_return = p_return_given_review / (df['stage4_return'].mean() * 100)
print(f"  Arrival lift (given Passage): {lift_arrival:.2f}x")
print(f"  Review lift (given Arrival): {lift_review:.2f}x")
print(f"  Return lift (given Review): {lift_return:.2f}x")

In [ ]:
# Visualization: Stage Transition Sankey-style
fig, ax = plt.subplots(figsize=(12, 6))

# Calculate flows
total = len(df)
s1 = df['stage1_passage'].sum()
s1_to_s2 = (df['stage1_passage'] & df['stage2_arrival']).sum()
s2_to_s3 = (df['stage2_arrival'] & df['stage3_review']).sum()
s3_to_s4 = (df['stage3_review'] & df['stage4_return']).sum()

# Create bar positions
stages = ['Total\nNDEs', 'Stage 1\nPassage', 'Stage 2\nArrival', 'Stage 3\nReview', 'Stage 4\nReturn']
counts = [total, s1, s1_to_s2, s2_to_s3, s3_to_s4]
colors = ['#2c7bb6', '#1a9850', '#91cf60', '#d9ef8b', '#fee08b']

bars = ax.bar(stages, counts, color=colors, edgecolor='black', linewidth=1.5)

# Add connecting lines
for i in range(len(counts)-1):
    ax.annotate('', xy=(i+1, counts[i+1]), xytext=(i, counts[i]),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# Add percentages
for i, (bar, count) in enumerate(zip(bars, counts)):
    if i == 0:
        label = f'{count:,}'
    else:
        pct = count / total * 100
        label = f'{count:,}\n({pct:.1f}%)'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
            label, ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Number of NDEs')
ax.set_title('Sequential Flow Through NDE Stages\n(Cumulative: Each stage requires previous stage)')
ax.set_ylim(0, max(counts) * 1.15)

plt.tight_layout()
plt.show()

---

## 5. Test 3: Stage Completeness Analysis

**Question**: How many experiences include all four stages? What characterizes complete vs incomplete journeys?

In [ ]:
print("="*70)
print("TEST 3: STAGE COMPLETENESS ANALYSIS")
print("="*70)

# Complete journey = all 4 stages
complete = (df['stages_present'] == 4).sum()
print(f"\n📊 Complete Journeys (all 4 stages): {complete:,} ({complete/len(df)*100:.1f}%)")

# Categorize completeness
df['completeness'] = pd.cut(df['stages_present'], 
                            bins=[-1, 0, 1, 2, 3, 4],
                            labels=['None', 'Minimal', 'Partial', 'Substantial', 'Complete'])

print("\n📊 Completeness Categories:")
comp_dist = df['completeness'].value_counts()
for cat in ['None', 'Minimal', 'Partial', 'Substantial', 'Complete']:
    if cat in comp_dist:
        count = comp_dist[cat]
        print(f"  {cat}: {count:,} ({count/len(df)*100:.1f}%)")

# Compare complete vs incomplete on key variables
print("\n📊 COMPLETE vs INCOMPLETE Comparison:")
complete_df = df[df['stages_present'] == 4]
incomplete_df = df[df['stages_present'] < 4]

print(f"\n  Sample sizes: Complete={len(complete_df):,}, Incomplete={len(incomplete_df):,}")

In [ ]:
# Compare total elements (depth proxy)
print("\n📊 Total Elements by Completeness:")
element_stats = df.groupby('completeness')['total_elements'].agg(['mean', 'std', 'count'])
print(element_stats.to_string())

# Kruskal-Wallis test
groups = [df[df['completeness'] == cat]['total_elements'].values 
          for cat in ['Minimal', 'Partial', 'Substantial', 'Complete'] 
          if cat in df['completeness'].values]
if len(groups) > 1:
    stat, p = kruskal(*groups)
    print(f"\n📐 Kruskal-Wallis test: H = {stat:.2f}, p = {p:.6f}")

# Valence comparison
print("\n📊 Overall Valence by Completeness:")
valence_crosstab = pd.crosstab(df['completeness'], df['overall_valence'], normalize='index') * 100
print(valence_crosstab.round(1).to_string())

---

## 6. Test 4: NDE Depth and Stage Completeness

**Hypothesis**: Deeper NDEs (more elements) correlate with more complete stage progression.

In [ ]:
print("="*70)
print("TEST 4: NDE DEPTH AND STAGE COMPLETENESS")
print("="*70)

# Correlation: total elements vs stages present
rho, p = spearmanr(df['total_elements'], df['stages_present'])
print(f"\n📐 Spearman correlation (elements vs stages): ρ = {rho:.3f}, p = {p:.6e}")

# By stage count
print("\n📊 Mean Total Elements by Stage Count:")
for n_stages in sorted(df['stages_present'].unique()):
    subset = df[df['stages_present'] == n_stages]
    mean_elements = subset['total_elements'].mean()
    print(f"  {n_stages} stages: mean {mean_elements:.2f} elements (n={len(subset):,})")

In [ ]:
# Visualization: Elements vs Stages
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
ax1 = axes[0]
df.boxplot(column='total_elements', by='stages_present', ax=ax1)
ax1.set_xlabel('Number of Stages Present')
ax1.set_ylabel('Total Elements')
ax1.set_title('Element Count by Stage Completeness')
plt.suptitle('')  # Remove automatic title

# Scatter with jitter
ax2 = axes[1]
jitter_stages = df['stages_present'] + np.random.normal(0, 0.1, len(df))
jitter_elements = df['total_elements'] + np.random.normal(0, 0.2, len(df))
ax2.scatter(jitter_stages, jitter_elements, alpha=0.3, s=20)
ax2.set_xlabel('Number of Stages Present')
ax2.set_ylabel('Total Elements')
ax2.set_title(f'Elements vs Stages (ρ = {rho:.3f})')

# Add regression line
z = np.polyfit(df['stages_present'], df['total_elements'], 1)
p_line = np.poly1d(z)
ax2.plot([0, 4], [p_line(0), p_line(4)], 'r--', linewidth=2, label='Trend')
ax2.legend()

plt.tight_layout()
plt.show()

---

## 7. Test 5: Return Decision Analysis

**Question**: What factors determine whether and how experiencers return?

**New Schema Advantage**: We now have separate fields for:
- `return_agency`: WHO decided (self, being, other, forced)
- `return_willingness`: WANTED to return or not
- `boundary_type`: What kind of barrier/boundary was encountered

In [ ]:
print("="*70)
print("TEST 5: RETURN DECISION ANALYSIS")
print("="*70)

# Filter to those with return memory
df_return = df[df['has_return']].copy()
print(f"\nAnalyzing {len(df_return):,} cases with return memory ({len(df_return)/len(df)*100:.1f}% of total)")

# Return agency distribution
print("\n📊 Return Agency (WHO decided):")
agency_dist = df_return['return_agency'].value_counts()
for val, count in agency_dist.items():
    print(f"  {val}: {count:,} ({count/len(df_return)*100:.1f}%)")

# Return willingness distribution
print("\n📊 Return Willingness (WANTED to return?):")
willing_dist = df_return['return_willingness'].value_counts()
for val, count in willing_dist.items():
    print(f"  {val}: {count:,} ({count/len(df_return)*100:.1f}%)")

# Boundary type distribution
print("\n📊 Boundary Type:")
boundary_dist = df_return['boundary_type'].value_counts()
for val, count in boundary_dist.items():
    print(f"  {val}: {count:,} ({count/len(df_return)*100:.1f}%)")

In [ ]:
# Cross-tabulation: Agency x Willingness
print("\n📊 CROSS-TABULATION: Return Agency × Willingness")
print("="*70)

# Filter out not_mentioned for cleaner analysis
df_return_clean = df_return[
    (df_return['return_agency'] != 'not_mentioned') & 
    (df_return['return_willingness'] != 'not_mentioned')
]

if len(df_return_clean) > 0:
    crosstab = pd.crosstab(df_return_clean['return_agency'], 
                           df_return_clean['return_willingness'],
                           margins=True)
    print(crosstab.to_string())
    
    # Key insight: Self-choice willingness profile
    self_choice = df_return_clean[df_return_clean['return_agency'] == 'self_choice']
    if len(self_choice) > 0:
        print(f"\n📍 SELF-CHOICE Returns (n={len(self_choice)}):")
        self_willing = self_choice['return_willingness'].value_counts(normalize=True) * 100
        for will, pct in self_willing.items():
            print(f"   {will}: {pct:.1f}%")
    
    # Being-directed willingness profile
    being_sent = df_return_clean[df_return_clean['return_agency'] == 'being_directed']
    if len(being_sent) > 0:
        print(f"\n📍 BEING-DIRECTED Returns (n={len(being_sent)}):")
        being_willing = being_sent['return_willingness'].value_counts(normalize=True) * 100
        for will, pct in being_willing.items():
            print(f"   {will}: {pct:.1f}%")

In [ ]:
# Visualization: Return patterns
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Agency distribution
ax1 = axes[0]
agency_counts = df_return['return_agency'].value_counts()
ax1.bar(agency_counts.index, agency_counts.values, color='steelblue')
ax1.set_title('Return Agency')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

# Willingness distribution
ax2 = axes[1]
willing_counts = df_return['return_willingness'].value_counts()
colors = {'willing': '#2E86AB', 'reluctant': '#E63946', 'mixed': '#F6AE2D', 
          'not_mentioned': '#CCCCCC', 'neutral': '#91cf60'}
bar_colors = [colors.get(x, '#999999') for x in willing_counts.index]
ax2.bar(willing_counts.index, willing_counts.values, color=bar_colors)
ax2.set_title('Return Willingness')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=45)

# Boundary type distribution
ax3 = axes[2]
boundary_counts = df_return['boundary_type'].value_counts()
ax3.bar(boundary_counts.index, boundary_counts.values, color='coral')
ax3.set_title('Boundary Type')
ax3.set_ylabel('Count')
ax3.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n✓ Return analysis complete")

---

## 8. Summary and Conclusions

In [ ]:
# Q6: Mission-Based Return Analysis
# =================================
print("="*70)
print("Q6: MISSION-BASED RETURN ANALYSIS")
print("="*70)

# First, check if return_reason field exists in our extraction
# Need to update our extraction function to include mission-related fields

def get_mission_status(record):
    """Determine if return was mission-based from return_integration."""
    return_data = record.get('analysis', {}).get('return_integration', {}) or {}
    return_reasons = return_data.get('return_reasons', [])
    
    if not isinstance(return_reasons, list):
        return_reasons = [return_reasons] if return_reasons else []
    
    # Check for mission-related reasons
    mission_keywords = ['earthly_mission', 'mission', 'purpose', 'task', 'assignment']
    
    for reason in return_reasons:
        if isinstance(reason, str):
            if any(kw in reason.lower() for kw in mission_keywords):
                return 'mission'
    
    # Check agency + willingness pattern for "sent back" signature
    agency = return_data.get('return_agency', 'not_mentioned')
    willingness = return_data.get('return_willingness', 'not_mentioned')
    
    if agency == 'being_directed' and willingness in ['reluctant', 'unwilling']:
        return 'potential_mission'  # Sent back against will = possible commissioning
    
    return 'non_mission'

# Re-extract with mission field
print("\n📊 Extracting mission status from original data...")

structured_dir = Path("../data/structured")
mission_data = []

for file in structured_dir.glob("*.json"):
    with open(file, 'r', encoding='utf-8') as f:
        record = json.load(f)
    
    mission_status = get_mission_status(record)
    
    # Get core elements for cross-pathway comparison
    analysis = record.get('analysis', {}) or {}
    passage = analysis.get('passage', {}) or {}
    separation = passage.get('separation', {}) or {}
    arrival = passage.get('arrival', {}) or {}
    review = analysis.get('life_review', {}) or {}
    return_int = analysis.get('return_integration', {}) or {}
    
    mission_data.append({
        'nde_code': record.get('title'),
        'mission_status': mission_status,
        'has_obe': separation.get('obe_occurrence', '') in ['yes_verified', 'yes_unverified'],
        'has_tunnel': passage.get('transition', {}).get('tunnel_experience', '') in ['classic_tunnel', 'tunnel_like'],
        'has_light': arrival.get('light_encounter', '') not in ['not_mentioned', 'no_light', ''],
        'has_beings': len(arrival.get('being_identifications', []) or []) > 0,
        'has_life_review': review.get('occurrence', '') in ['extensive', 'brief'],
        'return_agency': return_int.get('return_agency', 'not_mentioned'),
        'return_willingness': return_int.get('return_willingness', 'not_mentioned')
    })

df_mission = pd.DataFrame(mission_data)

# Mission status distribution
print("\n📈 Mission Status Distribution:")
mission_dist = df_mission['mission_status'].value_counts()
for status, count in mission_dist.items():
    pct = count / len(df_mission) * 100
    print(f"   {status}: {count:,} ({pct:.1f}%)")

# Highlight volunteer soul proportion
volunteer_count = len(df_mission[df_mission['mission_status'].isin(['mission', 'potential_mission'])])
volunteer_pct = volunteer_count / len(df_mission) * 100
print(f"\n🎯 VOLUNTEER SOUL PATH: {volunteer_count:,} NDEs ({volunteer_pct:.1f}%)")
print(f"   Expected range per model: 5-10%")

In [ ]:
# Q7: Mission Returns vs Return Willingness Pattern
# =================================================
print("="*70)
print("Q7: MISSION RETURNS vs RETURN WILLINGNESS")
print("="*70)
print("\nHypothesis: Mission returns correlate with involuntary/reluctant patterns")
print("           (Being 'sent back' against will supports commissioning theory)")

# Cross-tabulation: mission_status vs return_willingness
df_return_mission = df_mission[df_mission['return_willingness'] != 'not_mentioned']

crosstab = pd.crosstab(df_return_mission['mission_status'], 
                       df_return_mission['return_willingness'],
                       margins=True)
print("\n📊 Cross-Tabulation (Mission Status × Return Willingness):")
print(crosstab.to_string())

# Proportions within each mission status
print("\n📈 Return Willingness BY Mission Status:")
for status in ['mission', 'potential_mission', 'non_mission']:
    subset = df_return_mission[df_return_mission['mission_status'] == status]
    if len(subset) > 0:
        print(f"\n   {status.upper()} (n={len(subset)}):")
        willing_dist = subset['return_willingness'].value_counts(normalize=True) * 100
        for will, pct in willing_dist.items():
            print(f"      {will}: {pct:.1f}%")

# Chi-square test
print("\n📐 Chi-Square Test (Mission × Willingness):")
observed = pd.crosstab(df_return_mission['mission_status'], 
                       df_return_mission['return_willingness'])
if observed.shape[0] >= 2 and observed.shape[1] >= 2:
    chi2, p_val, dof, expected = chi2_contingency(observed)
    print(f"   χ² = {chi2:.2f}")
    print(f"   df = {dof}")
    print(f"   p-value = {p_val:.4f}")
    
    if p_val < 0.05:
        print(f"   ✅ SIGNIFICANT: Mission status IS associated with return willingness")
    else:
        print(f"   ⚠️ NOT SIGNIFICANT: Mission status NOT associated with willingness")
else:
    print("   ⚠️ Insufficient data for chi-square test")

In [ ]:
# Q8: Cross-Pathway Analysis - Core Phenomenology Independence
# ============================================================
print("="*70)
print("Q8: CROSS-PATHWAY ANALYSIS - Core Phenomenology Independence")
print("="*70)
print("\nHypothesis: Core NDE elements (OBE, tunnel, light, beings, life review)")
print("           should occur at SIMILAR rates across all pathways")
print("           (Mission is exceptional FEATURE, not a different EXPERIENCE)")

core_elements = ['has_obe', 'has_tunnel', 'has_light', 'has_beings', 'has_life_review']

print("\n📊 Core Element Rates by Pathway:")
print("-" * 60)
print(f"{'Element':<20} {'Non-Mission':>12} {'Mission':>12} {'Potential':>12}")
print("-" * 60)

for element in core_elements:
    non_mission = df_mission[df_mission['mission_status'] == 'non_mission']
    mission = df_mission[df_mission['mission_status'] == 'mission']
    potential = df_mission[df_mission['mission_status'] == 'potential_mission']
    
    non_rate = non_mission[element].mean() * 100 if len(non_mission) > 0 else 0
    mis_rate = mission[element].mean() * 100 if len(mission) > 0 else 0
    pot_rate = potential[element].mean() * 100 if len(potential) > 0 else 0
    
    print(f"{element:<20} {non_rate:>11.1f}% {mis_rate:>11.1f}% {pot_rate:>11.1f}%")

print("-" * 60)

# Chi-square tests for independence
print("\n📐 Chi-Square Independence Tests (Pathway × Core Element):")
print("   Testing whether core elements are INDEPENDENT of pathway type")
print()

independence_results = []
for element in core_elements:
    observed = pd.crosstab(df_mission['mission_status'], df_mission[element])
    if observed.shape[0] >= 2 and observed.shape[1] >= 2:
        chi2, p_val, dof, expected = chi2_contingency(observed)
        result = "INDEPENDENT" if p_val > 0.05 else "DEPENDENT"
        independence_results.append({
            'element': element,
            'chi2': chi2,
            'p_value': p_val,
            'result': result
        })
        print(f"   {element}: χ²={chi2:.2f}, p={p_val:.4f} → {result}")

# Summary
independent_count = sum(1 for r in independence_results if r['result'] == 'INDEPENDENT')
print(f"\n🎯 SUMMARY: {independent_count}/{len(independence_results)} core elements are INDEPENDENT of pathway")
print("   If most are independent → SUPPORTS 'universal phenomenology' claim")

---

## Part III: "Both/And" Resolution Validation

The Threefold Path model proposes that BOTH linear progression (normative) AND cyclical/mission-based returns (exceptional) are valid - but for DIFFERENT souls.

### Key Prediction:
- **90%+** follow normative linear path
- **5-10%** show mission-based (volunteer soul) pattern
- Restorative incarnation (not testable with NDE data)

In [ ]:
# Both/And Resolution Analysis
# ============================
print("="*70)
print("BOTH/AND RESOLUTION: Pathway Distribution")
print("="*70)

# Final pathway classification
df_mission['pathway'] = df_mission['mission_status'].apply(
    lambda x: 'VOLUNTEER' if x in ['mission', 'potential_mission'] else 'NORMATIVE'
)

pathway_dist = df_mission['pathway'].value_counts()
total = len(df_mission)

print("\n📊 FINAL PATHWAY DISTRIBUTION:")
print("=" * 40)
for pathway, count in pathway_dist.items():
    pct = count / total * 100
    bar = '█' * int(pct // 2)
    print(f"   {pathway:<12}: {count:>5,} ({pct:>5.1f}%) {bar}")
print("=" * 40)

# Compare to model predictions
normative_pct = pathway_dist.get('NORMATIVE', 0) / total * 100
volunteer_pct = pathway_dist.get('VOLUNTEER', 0) / total * 100

print("\n🎯 COMPARISON TO MODEL PREDICTIONS:")
print(f"   Normative Path:  {normative_pct:.1f}% (predicted: 90%+)")
print(f"   Volunteer Path:  {volunteer_pct:.1f}% (predicted: 5-10%)")

if normative_pct >= 90:
    print("\n   ✅ NORMATIVE PATH: Prediction VALIDATED (≥90%)")
else:
    print(f"\n   ⚠️ NORMATIVE PATH: Below prediction ({normative_pct:.1f}% < 90%)")

if 5 <= volunteer_pct <= 10:
    print("   ✅ VOLUNTEER PATH: Prediction VALIDATED (5-10%)")
elif volunteer_pct > 10:
    print(f"   ⚠️ VOLUNTEER PATH: Above prediction ({volunteer_pct:.1f}% > 10%)")
else:
    print(f"   ⚠️ VOLUNTEER PATH: Below prediction ({volunteer_pct:.1f}% < 5%)")

print("\n📖 INTERPRETATION:")
print("   The 'Both/And' model proposes that BOTH linear and cyclical views are valid,")
print("   but they apply to DIFFERENT populations:")
print("   - Linear progression is NORMATIVE for most souls")
print("   - Mission-based returns are EXCEPTIONAL (volunteer souls)")
print("   - This resolves the apparent contradiction in afterlife philosophies")

In [ ]:
print("="*70)
print("SUMMARY: Threefold Path Validation")
print("="*70)

print("""
┌─────────────────────────────────────────────────────────────────────┐
│ TEST                        │ RESULT                               │
├─────────────────────────────┼──────────────────────────────────────┤
│ 1. Stage Occurrence         │ [PENDING - see analysis above]       │
│    - All 4 stages present   │                                      │
│      in dataset?            │                                      │
├─────────────────────────────┼──────────────────────────────────────┤
│ 2. Sequential Order         │ [PENDING - see analysis above]       │
│    - Stages follow          │                                      │
│      predicted sequence?    │                                      │
├─────────────────────────────┼──────────────────────────────────────┤
│ 3. Completeness             │ [PENDING - see analysis above]       │
│    - How many complete      │                                      │
│      journeys?              │                                      │
├─────────────────────────────┼──────────────────────────────────────┤
│ 4. Depth Correlation        │ [PENDING - see analysis above]       │
│    - Deeper NDEs = more     │                                      │
│      stages?                │                                      │
├─────────────────────────────┼──────────────────────────────────────┤
│ 5. Return Patterns          │ [PENDING - see analysis above]       │
│    - Agency and willingness │                                      │
│      patterns?              │                                      │
└─────────────────────────────┴──────────────────────────────────────┘

NOTE: Run all cells above to populate results.
""")

print(f"\nDataset: {len(df):,} NDE records")
print(f"Schema: Updated with split return fields (agency, willingness, boundary)")